# MeanFlow Notebook

This notebook combines dataset loading, model definition, training, and one-step generation evaluation in a single file.

In [ ]:
# !pip install -q torch torchvision matplotlib numpy tqdm


from google.colab import drive
import os

drive.mount('/content/drive')

DATA_ROOT = '/content/drive/MyDrive/MeanFlow/imagenette_subset'
train_dataset_path = f'{DATA_ROOT}/train_npy/'
val_dataset_path = f'{DATA_ROOT}/val_npy/'

print('DATA_ROOT:', DATA_ROOT)
print('train exists:', os.path.exists(train_dataset_path))
print('val exists:', os.path.exists(val_dataset_path))
if not os.path.exists(train_dataset_path):
    raise FileNotFoundError(f'Missing path: {train_dataset_path}')

Mounted at /content/drive
DATA_ROOT: /content/drive/MyDrive/MeanFlow/imagenette_subset
train exists: True
val exists: True


In [13]:
import os
import time
import math
import random

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, Subset
from tqdm import tqdm

In [14]:
# ---- Helpers ----

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def plot_loss(train_loss, val_loss, save_path):
    plt.figure()
    plt.plot(train_loss, label="train")
    plt.plot(val_loss, label="eval")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(save_path, "loss.png"))
    plt.close()


def create_experiment_dir(experiment_num: int, root_dir: str = "experiments") -> str:
    save_path = os.path.join(root_dir, f"run{experiment_num}")
    os.makedirs(save_path, exist_ok=True)
    return save_path


class MeanFlowDataset(Dataset):
    """Load preprocessed 64x64x3 images from train_npy/{class}_npy/*.npy."""

    def __init__(self, data_dir: str):
        class_dirs = sorted(
            d
            for d in os.listdir(data_dir)
            if os.path.isdir(os.path.join(data_dir, d)) and d.endswith("_npy")
        )
        self.samples = [
            (os.path.join(data_dir, class_dir, filename), class_idx)
            for class_idx, class_dir in enumerate(class_dirs)
            for filename in sorted(os.listdir(os.path.join(data_dir, class_dir)))
            if filename.endswith(".npy")
        ]

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, index: int):
        path, label = self.samples[index]
        image = np.load(path)
        image = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0
        return image, label

In [15]:
# ---- UNet + timestep helpers ----

def sample_t_r(batch_size, percent_unequal=0.25):
    t = torch.rand(batch_size)
    r = torch.rand(batch_size) * t

    num_unequal = max(int(percent_unequal * batch_size), 2)
    unequal_mask = torch.zeros(batch_size, dtype=torch.bool)
    unequal_indices = torch.randperm(batch_size)[:num_unequal]
    unequal_mask[unequal_indices] = True

    r = torch.where(unequal_mask, r, t)
    return t, r


def timestep_emb(timesteps, d_emb):
    half_dim = d_emb // 2
    # Keep all tensors on the same device as timesteps (important for Colab GPU).
    freqs = torch.exp(
        -math.log(10000)
        * torch.arange(half_dim, device=timesteps.device, dtype=torch.float32)
        / (half_dim - 1)
    )
    args = timesteps[:, None].float() * freqs[None, :]

    emb = torch.zeros(
        len(timesteps),
        d_emb,
        device=timesteps.device,
        dtype=args.dtype,
    )
    emb[:, 0::2] = torch.sin(args)
    emb[:, 1::2] = torch.cos(args)
    return emb


def create_timestep_embs(r, t, d_emb):
    t_embs = timestep_emb(t, d_emb)
    tr_embs = timestep_emb(t - r, d_emb)
    return t_embs, tr_embs


class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, d_emb=256):
        super().__init__()
        self.proj_embs = nn.Linear(d_emb, out_ch)
        self.norm1 = nn.GroupNorm(8, in_ch)
        self.act1 = nn.LeakyReLU()
        self.conv1 = nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1)
        self.norm2 = nn.GroupNorm(8, out_ch)
        self.act2 = nn.LeakyReLU()
        self.conv2 = nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1)
        self.skip = nn.Identity() if in_ch == out_ch else nn.Conv2d(in_ch, out_ch, kernel_size=1)

    def forward(self, x, tembs):
        tembs = self.proj_embs(tembs)
        tembs = tembs[:, :, None, None]

        f = self.norm1(x)
        f = self.act1(f)
        f = self.conv1(f)
        f = f + tembs
        f = self.norm2(f)
        f = self.act2(f)
        f = self.conv2(f)
        return f + self.skip(x)


class UNet(nn.Module):
    def __init__(self, in_ch=3, ch=(64, 128, 256, 512), d_emb=256):
        super().__init__()
        self.d_emb = d_emb

        self.mlp_t = nn.Sequential(
            nn.Linear(d_emb, d_emb),
            nn.SiLU(),
            nn.Linear(d_emb, d_emb),
        )
        self.mlp_tr = nn.Sequential(
            nn.Linear(d_emb, d_emb),
            nn.SiLU(),
            nn.Linear(d_emb, d_emb),
        )

        self.in_project = nn.Conv2d(in_ch, ch[0], kernel_size=3, padding=1)
        self.enc1 = ResBlock(ch[0], ch[1], d_emb)
        self.pool1 = nn.MaxPool2d(2)

        self.enc2 = ResBlock(ch[1], ch[2], d_emb)
        self.pool2 = nn.MaxPool2d(2)

        self.enc3 = ResBlock(ch[2], ch[3], d_emb)
        self.pool3 = nn.MaxPool2d(2)

        self.bottleneck = ResBlock(ch[3], ch[3], d_emb)

        self.up1 = nn.ConvTranspose2d(ch[3], ch[3], kernel_size=2, stride=2)
        self.dec1 = ResBlock(ch[3] * 2, ch[2], d_emb)

        self.up2 = nn.ConvTranspose2d(ch[2], ch[2], kernel_size=2, stride=2)
        self.dec2 = ResBlock(ch[2] * 2, ch[1], d_emb)

        self.up3 = nn.ConvTranspose2d(ch[1], ch[1], kernel_size=2, stride=2)
        self.dec3 = ResBlock(ch[1] * 2, ch[0], d_emb)
        self.out_proj = nn.Conv2d(ch[0], in_ch, kernel_size=3, padding=1)

    def forward(self, x, r, t):
        t_embs, tr_embs = create_timestep_embs(r, t, self.d_emb)
        t_embs = self.mlp_t(t_embs)
        tr_embs = self.mlp_tr(tr_embs)
        tembs = t_embs + tr_embs

        x = self.in_project(x)
        x = self.enc1(x, tembs)
        skip1 = x
        x = self.pool1(x)

        x = self.enc2(x, tembs)
        skip2 = x
        x = self.pool2(x)

        x = self.enc3(x, tembs)
        skip3 = x
        x = self.pool3(x)

        x = self.bottleneck(x, tembs)

        x = self.up1(x)
        x = torch.cat([x, skip3], dim=1)
        x = self.dec1(x, tembs)

        x = self.up2(x)
        x = torch.cat([x, skip2], dim=1)
        x = self.dec2(x, tembs)

        x = self.up3(x)
        x = torch.cat([x, skip1], dim=1)
        x = self.dec3(x, tembs)
        x = self.out_proj(x)
        return x

    def u_fn(self, z, r, t):
        return self.forward(z, r, t)

In [16]:
# ---- Config ----
run = 7
seed = 42
epochs = 10000
lr = 1e-3
eta_min = 1e-5
batch_size = 3
overfit_indices = [0, 1, 2]
num_generated = 1024

# Save locally to Colab runtime experiments folder.
LOCAL_EXPERIMENTS_ROOT = "/content/experiments"

# Optional: copy results to Drive after training finishes.
COPY_RESULTS_TO_DRIVE = True
DRIVE_EXPERIMENTS_ROOT = "/content/drive/MyDrive/MeanFlow/experiments"

# Paths are set in cell 1 from Google Drive mount.
# train_dataset_path should point to .../imagenette_subset/train_npy/

set_seed(seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
print("Train path:", train_dataset_path)

if not os.path.exists(train_dataset_path):
    raise FileNotFoundError(
        f"Missing path: {train_dataset_path}. Run cell 1 first and verify DATA_ROOT."
    )

save_path = create_experiment_dir(run, LOCAL_EXPERIMENTS_ROOT)
train_dataset = MeanFlowDataset(train_dataset_path)
train_subset = Subset(train_dataset, overfit_indices)
small_train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=False)
small_eval_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=False)

print("Local save path:", save_path)
print("num train samples:", len(train_dataset))
print("subset size:", len(train_subset))
x0, y0 = train_subset[0]
print("sample shape:", tuple(x0.shape), "label:", y0)

Using device: cuda
Train path: /content/drive/MyDrive/MeanFlow/imagenette_subset/train_npy/
Local save path: /content/experiments/run7
num train samples: 3
subset size: 3
sample shape: (3, 64, 64) label: 0


In [ ]:
# ---- Train ----
loss_fn = nn.MSELoss()
model = UNet(in_ch=3, ch=(64, 128, 256, 512), d_emb=256).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.0)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=eta_min)

train_start_time = time.time()
train_losses, eval_losses = [], []
best_train_loss = float("inf")

print(f"Training experiment {run}")
for epoch in range(epochs):
    model.train()
    epoch_start_time = time.time()
    epoch_train_loss = 0.0

    for x, _ in small_train_loader:
        x = x.to(device)
        optimizer.zero_grad()

        t, r = sample_t_r(x.shape[0])
        t = t.to(device)
        r = r.to(device)

        t_reshape = t.view(-1, 1, 1, 1)
        r_reshape = r.view(-1, 1, 1, 1)

        e = torch.randn_like(x)
        z = (1 - t_reshape) * x + t_reshape * e
        v = e - x

        u, dudt = torch.func.jvp(
            model.u_fn,
            (z, r, t),
            (v, torch.zeros_like(r), torch.ones_like(t)),
        )
        u_tgt = v - (t_reshape - r_reshape) * dudt

        train_loss = loss_fn(u, u_tgt.detach())
        train_loss.backward()
        optimizer.step()
        epoch_train_loss += train_loss.item()

    epoch_train_loss /= len(small_train_loader)
    train_losses.append(epoch_train_loss)

    model.eval()
    epoch_eval_loss = 0.0
    for x, _ in small_eval_loader:
        x = x.to(device)

        t, r = sample_t_r(x.shape[0])
        t = t.to(device)
        r = r.to(device)

        t_reshape = t.view(-1, 1, 1, 1)
        r_reshape = r.view(-1, 1, 1, 1)

        e = torch.randn_like(x)
        z = (1 - t_reshape) * x + t_reshape * e
        v = e - x

        with torch.no_grad():
            u, dudt = torch.func.jvp(
                model.u_fn,
                (z, r, t),
                (v, torch.zeros_like(r), torch.ones_like(t)),
            )
        u_tgt = v - (t_reshape - r_reshape) * dudt
        eval_loss = loss_fn(u, u_tgt.detach())
        epoch_eval_loss += eval_loss.item()

    epoch_eval_loss /= len(small_eval_loader)
    eval_losses.append(epoch_eval_loss)
    scheduler.step()

    if epoch_train_loss < best_train_loss:
        best_train_loss = epoch_train_loss
        torch.save(model.state_dict(), os.path.join(save_path, "meanflow_best.pt"))

    if epoch == 0 or (epoch + 1) % 25 == 0:
        plot_loss(train_losses, eval_losses, save_path)

    print(
        f"Epoch {epoch + 1}: {(time.time() - epoch_start_time) / 60:.2f} min | "
        f"Train: {epoch_train_loss:.6f} | Eval: {epoch_eval_loss:.6f}"
    )

torch.save(model.state_dict(), os.path.join(save_path, "meanflow.pt"))
print(f"Best train loss: {best_train_loss:.6f}")
print(f"Total training time: {(time.time() - train_start_time) / 60:.2f} min")
print(f"Saved locally to: {save_path}")

if COPY_RESULTS_TO_DRIVE:
    import shutil

    drive_save_path = create_experiment_dir(run, DRIVE_EXPERIMENTS_ROOT)
    shutil.copytree(save_path, drive_save_path, dirs_exist_ok=True)
    print(f"Copied results to Drive: {drive_save_path}")

Training experiment 7
Epoch 1: 0.03 min | Train: 1.119716 | Eval: 3.222712
Epoch 2: 0.00 min | Train: 3.171169 | Eval: 1.149191
Epoch 3: 0.00 min | Train: 1.139280 | Eval: 1.366416
Epoch 4: 0.00 min | Train: 1.421018 | Eval: 1.004567
Epoch 5: 0.00 min | Train: 1.017536 | Eval: 0.987463
Epoch 6: 0.00 min | Train: 0.882394 | Eval: 0.988963
Epoch 7: 0.01 min | Train: 0.870421 | Eval: 0.894368
Epoch 8: 0.01 min | Train: 0.854728 | Eval: 0.699010
Epoch 9: 0.00 min | Train: 0.836824 | Eval: 0.581805
Epoch 10: 0.00 min | Train: 0.585453 | Eval: 0.649963
Epoch 11: 0.00 min | Train: 0.748963 | Eval: 0.610357
Epoch 12: 0.00 min | Train: 0.508990 | Eval: 0.597524
Epoch 13: 0.00 min | Train: 0.580975 | Eval: 0.352206
Epoch 14: 0.00 min | Train: 0.567789 | Eval: 0.405434
Epoch 15: 0.00 min | Train: 0.402414 | Eval: 0.332311
Epoch 16: 0.00 min | Train: 0.473691 | Eval: 0.323712
Epoch 17: 0.00 min | Train: 0.297952 | Eval: 0.415057
Epoch 18: 0.00 min | Train: 0.251293 | Eval: 0.452444
Epoch 19: 0.00 

In [ ]:
# ---- One-step generation eval (nearest-match MSE over many samples) ----
checkpoint_path = os.path.join(save_path, "meanflow_best.pt")
model = UNet(in_ch=3, ch=(64, 128, 256, 512), d_emb=256).to(device)
model.load_state_dict(torch.load(checkpoint_path, map_location=device, weights_only=True))
model.eval()

x, labels = next(iter(small_eval_loader))
x = x.to(device)
noise_generator = torch.Generator(device=device).manual_seed(seed + 1)

with torch.no_grad():
    generated_batches = []
    for start in tqdm(range(0, num_generated, batch_size), desc="Generating one-step samples"):
        current_batch_size = min(batch_size, num_generated - start)
        e = torch.randn(
            current_batch_size,
            x.shape[1],
            x.shape[2],
            x.shape[3],
            device=device,
            generator=noise_generator,
        )
        r = torch.zeros(current_batch_size, device=device)
        t = torch.ones(current_batch_size, device=device)
        x_gen = (e - model(e, r, t)).clamp(0.0, 1.0)
        generated_batches.append(x_gen.cpu())

gt = x.detach().cpu()
all_generated = torch.cat(generated_batches, dim=0)

mse_matrix = ((gt[:, None] - all_generated[None, :]) ** 2).mean(dim=(2, 3, 4))
best_mse, best_idx = mse_matrix.min(dim=1)

fig, axes = plt.subplots(2, 3, figsize=(10, 6))
gt_images = gt.permute(0, 2, 3, 1).numpy()
best_images = all_generated[best_idx].permute(0, 2, 3, 1).numpy()

for i, label in enumerate(labels.tolist()):
    axes[0, i].imshow(gt_images[i])
    axes[0, i].set_title(f"GT (label {label})")
    axes[0, i].axis("off")

    axes[1, i].imshow(best_images[i])
    axes[1, i].set_title(f"Best sample MSE={best_mse[i].item():.6f}")
    axes[1, i].axis("off")

plt.tight_layout()
plt.savefig(os.path.join(save_path, "samples_colab.png"))
plt.show()

for i, mse in enumerate(best_mse.tolist()):
    print(f"Image {i} best reconstruction MSE over {num_generated} samples: {mse:.8f}")